# Supabase Deletion Plan — open-platform & open-operations

**Goal:** shrink both Supabase projects to fit the free tier by deleting ephemeral/log/history data **after** it is safely backed up to Databricks and Docker Hub. The two apps must keep working on their live tables.

## Verified backup state (2026-11-21)

| Backup | Contents | Location | Verified |
| --- | --- | --- | --- |
| Full mirror (Delta) | 296 tables, 240,222 source rows — `open_platform` 174/174, `open_operations` 113/122 canonical + 9 quota-blocked (fresh copies in `open_ops_mirror`) | Databricks | 287/287 row-count parity, 0 mismatches (re-verified 2026-11-21; hourly re-check automation running) |
| Raw JSONL backup + manifest | 296 tables, ~88 MB, SHA-256 per file | `agent-brain-backup/20261121T032243Z` in the raw_files volume (backup taken 2026-11-21 03:22 UTC) | manifest + spot SHA-256 verified |
| Full Parquet export + tar.gz (superseded) | 143-table era, 27.18 MB | `/Volumes/open_connect/supabase_migration/raw_files/` | copied, size checked — superseded by the 296-table backup |
| Public-safe subset (28 tables, 158,813 rows) | 10.73 MB image layer | `hillstreet/open-connect-db-archive:2026-11-21` (public) | anonymous pull + digest match |
| Sensitive tables (auth, secrets, private messages) | in full mirror only | Databricks volume + Delta | not published publicly |

**Mirror freshness caveat:** the mirror now reflects the 2026-11-21 03:22 UTC backup with verified row parity, but live Supabase data has changed since — the Phase 1 fresh re-export remains **mandatory**. Note: an automated external sync periodically rewrites mirror tables (and recreates a `shadow` schema of backup tables); the hourly parity automation flags any drift it causes.

**Size caveat:** the sizes above are Databricks/Delta-compressed. They do **not** equal Supabase on-disk sizes (Postgres JSONB + indexes + bloat are typically several times larger). Phase 0 measurement decides what actually needs to go.

## Free-tier targets (verify current limits in your Supabase dashboard → Settings → Billing)

Free plan historically includes ~500 MB database size, ~1 GB file storage, and pauses projects after ~1 week of inactivity. Confirm the exact current numbers in the dashboard before setting the target — the plan optimizes to "comfortably under the DB limit", not to a hardcoded number.

## Prerequisites (currently missing — nothing runs until these exist)

1. **Supabase access** — one of:
   * Postgres connection string per project (Settings → Database → Connection string / pooler), or
   * Supabase Management API key (supabase.com dashboard → Account → Tokens) to measure sizes via API.
   * Do **not** paste secrets into this notebook — pass them as environment variables or Databricks secret scopes when running.
2. **psycopg2** on the compute (for direct PG execution) or use the Supabase SQL editor for the measurement queries.
3. **Docker Hub PAT** — the PAT pasted earlier in this workstream has been rotated and now fails auth (verified 2026-11-21: 401). A **fresh PAT** is required for the Phase 1 private backup push; the free private-repo slot remains available.

## Safety gates (hard requirements)

* **Gate 1 — Fresh backup:** Phase 1 completes with row parity between Supabase and the Databricks mirror for every table in the deletion list.
* **Gate 2 — Explicit approval:** each phase's generated SQL is reviewed and approved in chat before execution. No `TRUNCATE`/`DELETE` runs without it.
* **Gate 3 — Verification between phases:** after each phase, re-measure size and confirm the apps still work (open-connect.site health, tgate worker heartbeat).
* **Rollback:** deleted data is restorable from the Parquet/tar.gz backups (insert back via psycopg2 from the volume). Session cleanup logs all users out — schedule it for a quiet window.

In [0]:
# Verified inventory + curated classification (source: fresh 296-table backup manifest, 2026-11-21 03:22 UTC)
# Re-derived 2026-11-21: curated sets unchanged. The 130 tables NOT covered by the previous
# 121-table export are ALL EMPTY at source (0 rows as of this backup, verified) — they fall
# into KEEP by default, contribute nothing to DB size, and are re-checked in the Phase 1 re-export.
# The 9 quota-blocked open-operations tables (projects, workspaces, organizations, etc.) live in
# open_connect.open_ops_mirror with fresh data; their counts below come from this backup manifest.
import json

with open("/Volumes/open_connect/supabase_migration/raw_files/agent-brain-backup/20261121T032243Z/manifest.json") as f:
    M = json.load(f)

SCHEMA_OF = {"open-platform": "open_platform", "open-operations": "open_operations"}
rows = {}
for proj, pdata in M["projects"].items():
    for e in pdata["entries"]:
        rows[f"open_connect.{SCHEMA_OF[proj]}.{e['schema']}__{e['table']}"] = e["exported"]

def r(schema, table):
    return rows.get(f"open_connect.{schema}.{table}", 0)

# DELETE — logs / ephemeral / history; all present in the 2026-11-21 backup
DELETE = {
  "open_platform": {
    "cron__job_run_details", "net___http_response",
    "auth__audit_log_entries", "public__gateway_requests",
    "public__autonomous_runs", "public__autonomous_run_events",
    "public__workforce_events", "public__open_system_sync_events",
    "public__open_system_deployments", "public__open_system_backup_catalog",
    "public__open_system_backup_lifecycle", "public__open_system_backup_verified_parts",
    "public__control_audit_events", "public__data_control_activity",
    "public__agent_chats", "public__memory_records", "public__knowledge_items",
    "public__x_search_nodes", "public__x_task_items", "public__x_storages",
    "public__databricks_sync_audit", "public__e2e_autonomous_verification",
    "public__hillstreet_otp_calls", "public__hillstreet_otp_jobs",
    "public__hillstreet_otp_number_inventory", "public__hillstreet_otp_runtime",
    "public__hillstreet_sms_deliveries", "public__hillstreet_sms_forwarding",
    "public__hillstreet_website_inquiries",
  },
  "open_operations": {
    "public__open_tgate_tg_messages", "public__open_tgate_audit_events",
    "public__open_tgate_workspace_audit_events", "public__open_tgate_login_commands",
    "public__open_tgate_oauth_refresh", "public__open_tgate_worker_heartbeats",
    "auth__audit_log_entries",  # exists in BOTH projects (161 rows in operations mirror)
    "public__databricks_sync_audit", "private__storage_usage_snapshots",
  },
}

# SESSION CLEANUP — Supabase-managed auth sessions; deleting forces re-login everywhere
SESSION_CLEANUP = {
  "auth__sessions", "auth__refresh_tokens", "auth__flow_state",
  "auth__one_time_tokens", "auth__mfa_amr_claims",
}

# REVIEW — decide per table after Phase 0 sizing (kept by default)
REVIEW = {
  "open_platform": {
    "storage__objects": "orphaned/expired files only; must delete files via Storage API too",
    "public__resources": "keep if catalog is live; trim old rows otherwise",
    "public__resource_source_records": "crawl history — trim if resources trimmed",
    "public__workforce_agent_versions": "keep latest N versions per agent",
    "public__x_users": "PII — keep (live app state)",
    "public__x_setting_items": "keep (may hold live settings)",
  },
  "open_operations": {
    "public__open_tgate_tg_chats": "keep if tgate worker is live (active chat state)",
    "public__open_tgate_tg_entities": "keep if tgate worker is live",
    "public__open_tgate_tg_accounts": "keep — linked accounts",
    "public__open_tgate_knowledge_sources": "keep if tgate knowledge used",
  },
}

# everything else = KEEP (live app data: auth users/identities/mfa, vault/credential secrets,
# api keys, oauth clients, orgs/workspaces/projects, profiles, toolkits, registries, buckets,
# and the 130 empty-at-source tables — 0 rows as of the 03:22 UTC backup, no size impact)
for s in ("open_platform", "open_operations"):
    tables = {t.split("open_connect.")[1].split(".", 1)[1] for t in rows if t.startswith(f"open_connect.{s}.")}
    keep = tables - DELETE[s] - set(REVIEW[s]) - SESSION_CLEANUP
    print(f"{s}: {len(DELETE[s])} DELETE, {len(SESSION_CLEANUP)} session-cleanup, {len(REVIEW[s])} review, {len(keep)} KEEP")
    print(f"  DELETE rows in backup: {sum(r(s, t) for t in DELETE[s]):,}")
    print(f"  KEEP  rows in backup: {sum(r(s, t) for t in keep):,}")

## Phase 0 — Measure live sizes (Supabase side)

Run `MEASURE_SQL` in each project's Supabase SQL editor (or via psycopg2 once credentials exist). This produces:

* total database size (`pg_database_size`),
* per-table on-disk size + row estimate (`pg_total_relation_size`), sorted descending,
* storage bucket usage (per bucket byte totals).

Compare the live per-table sizes against the mirror classification above. Expected outcome: the log/history tables dominate on-disk size (they usually carry JSONB payloads + indexes). If the total is already under the free-tier limit, **stop here** — no deletion needed.

**Decision rule:** after Phase 0, re-check the REVIEW tables — if the DELETE set alone frees enough to fit the limit, leave REVIEW tables untouched.

In [0]:
# Phase 0 — run this SQL in EACH Supabase project's SQL editor (not runnable against Databricks)
MEASURE_SQL = """
-- total DB size
SELECT pg_size_pretty(pg_database_size(current_database())) AS db_size;

-- per-table size, descending
SELECT n.nspname AS schema_name,
       c.relname AS table_name,
       pg_size_pretty(pg_total_relation_size(c.oid)) AS total_size,
       pg_total_relation_size(c.oid) AS total_bytes,
       c.reltuples::bigint AS est_rows
FROM pg_class c
JOIN pg_namespace n ON n.oid = c.relnamespace
WHERE c.relkind = 'r'
  AND n.nspname NOT IN ('pg_catalog', 'information_schema', 'pg_toast', '_timescaledb_catalog', '_timescaledb_internal')
ORDER BY total_bytes DESC;

-- storage usage per bucket
SELECT bucket_id, count(*) AS objects, pg_size_pretty(sum(size)) AS total
FROM storage.objects GROUP BY bucket_id ORDER BY sum(size) DESC;
"""
print(MEASURE_SQL)

## Phase 1 — Fresh backup + parity verification (Gate 1)

**Current blocker (2026-11-21):** the org hit its Storage quota — the dashboard reports 685.5 GB / 1 GB org-wide — and **all open-platform services now return 402 (restricted)**, including REST, Storage API, and S3. Verified by direct probe. The 03:22 UTC backup predates the restriction, so the DB snapshot is current; the DB itself is only ~0.051 GB. This Phase 1 re-export cannot run for open-platform until restrictions lift (upgrade / spend-cap removal / storage cleanup). The dashboard's 685 GB cannot be explained by live objects (metadata shows ~13 MB) — it is almost certainly orphaned backend blobs from the backup pipelines; note that a billing-period reset alone will not remove stored bytes.

**Live status update (2026-11-23, verified via Supabase MCP read-only SQL — `mcp.supabase.com/mcp?project_ref=...&read_only=true` with the account PAT bypasses the 402):**

* **A source-side purge ran between Nov 21 03:22 and Nov 23**, closely matching this plan's DELETE sets. open-platform: 12 tables zeroed (gateway_requests 31,968→0; open_system_backup_verified_parts 82,791→0; open_system_backup_catalog 1,450→0; open_system_backup_lifecycle 1,548→0; resources 969→0; toolkit_items 1,121→0; autonomous_runs; autonomous_run_events; databricks_sync_audit; open_system_sync_events; project_resources; resource_source_records) + cron.job_run_details trimmed 65,067→13,817. open-operations: open_tgate history zeroed (tg_messages 29,694→0; tg_entities 4,365→0; tg_chats 611→0; login_commands 32→0), audit_events 11,782→611, cron trimmed 28→9.
* **The Databricks mirror is now the ONLY surviving copy of those purged rows** — verified 10/10 exact spot counts (gateway_requests 31,968; verified_parts 82,791; tgate tg_messages 29,694; audit_events 11,782; etc.). Do NOT trim or drop canonical mirror tables to "match live" — the mirror is the preservation copy by design.
* **Post-backup additions captured**: 10 new `open_box` schema tables on open-platform (AList-style file browser: x_setting_items 116 rows, x_task_items 4, x_users 2, rest empty) exported as JSONL with sha256 manifest to `/Volumes/open_connect/supabase_migration/raw_files/live_extras/20261123T051559Z/`. Materializing them as UC tables in `open_connect.live_extras` is **blocked by the metastore table quota (583 estimated / 500 limit — worse than Nov 21's ~561; the account-team raise request is now urgent)**. Also captured: public.x_storages 1 row (open-platform), private.storage_usage_snapshots full 31 rows (open-operations).
* open-operations storage has been **0 bytes all billing cycle** per its usage snapshots (all 6 buckets empty every day Nov 16–23) — it is not a source of the 685 GB.

The Databricks mirror reflects the 2026-11-21 03:22 UTC backup (296 tables; 287/287 canonical row parity + 9 quota-blocked copies refreshed in `open_ops_mirror`), but live Supabase data has changed since then. Before any deletion:

1. **Re-export** every table in the DELETE + SESSION_CLEANUP + REVIEW sets from the live Supabase projects into a fresh dated folder under the volume (same Parquet layout as the 2026-11-21 export).
2. **Parity check:** for each table, `count(*)` in Supabase must match or exceed the mirror count (live only grows between syncs). A live count **lower** than the mirror means an assumption is wrong — stop and investigate. **Exception:** the SESSION_CLEANUP tables (sessions, refresh tokens, flow_state, one-time tokens) expire and rotate continuously, so a live count below the mirror is *expected* for them — treat it as a warning, not a stop.
3. **Push fresh full backup** to a new **private** Docker Hub repo (the free private slot is available now) — full archive including sensitive tables this time, since it is private. Requires a fresh Docker Hub PAT: the earlier one was rotated and no longer authenticates (see Prerequisites).
4. Record the fresh manifest in the volume next to the old one.

Rollback source of truth after deletion = this fresh export, not the Nov 21 one.

In [0]:
# Phase 1 skeleton — fill in connection details via env vars, never paste secrets in the notebook
# Requires: %pip install psycopg2-binary

import os

# SUPA_PLATFORM_URL = os.environ["SUPA_PLATFORM_URL"]     # postgres://...pooler.supabase.com:6543/postgres
# SUPA_OPERATIONS_URL = os.environ["SUPA_OPERATIONS_URL"]

# For each (project, schema) pair in DELETE / SESSION_CLEANUP / REVIEW:
#   1) SELECT * -> write Parquet to
#      /Volumes/open_connect/supabase_migration/raw_files/fresh_export_<date>/<schema>/<table>
#      Connection: use the direct/session connection (db.<ref>.supabase.co:5432 or the session
#      pooler) — named server-side cursors do NOT survive the transaction pooler (:6543); with
#      :6543 use a plain cursor + fetchmany() batches instead
#   2) record live row counts
#   3) compare to the mirror counts (cell 3) — parity gate
#   4) tar.gz + push to private Docker Hub repo (reuse the registry-API push flow that
#      created hillstreet/open-connect-db-archive; set is_private=true this time)

print("Skeleton — activate when Supabase credentials are provided in chat.")

## Phase 2 — Phased deletion (Gate 2: approval before each phase)

Deletion runs **per project, in phases**, with size re-measurement and an app smoke check between phases. `DELETE FROM` (not `TRUNCATE`) is used to avoid lock escalation and FK surprises; the cell below generates the exact statements.

| Phase | Target | Tables (mirror row counts) | Notes |
| --- | --- | --- | --- |
| A | logs & audit | gateway_requests, *_audit_events, workforce_events, sync/deploy events, cron run details, http cache, heartbeats, storage snapshots, auth audit logs (both projects) | zero app impact |
| B | history content | agent_chats, memory_records, knowledge_items, autonomous_runs(+events), x_* task/search data, hillstreet OTP/SMS/inquiries, backup catalog/lifecycle/parts, tgate messages, tgate login commands, tgate oauth refresh | app loses history UI; live state intact |
| C | auth sessions | auth.sessions, refresh_tokens, flow_state, one_time_tokens, mfa_amr_claims | **logs all users out** — quiet window |
| D | REVIEW tables | only if Phase 0 shows the DB still over limit | case-by-case, per the REVIEW notes |

**Kept untouched (live app data):** auth users/identities/mfa_factors, vault & credential secrets, api_keys, oauth clients, orgs/workspaces/memberships, projects, profiles, user_roles, toolkits/categories/registries, tgate accounts/chats/entities/operators, buckets, cron jobs, schedules, tasks, environments.

In [0]:
# Phase 2 — generates the exact DELETE statements per project/phase.
# REVIEW BEFORE RUNNING — nothing executes here; output is the SQL to approve and run in Supabase.

def to_pg(name):
    """mirror name (schema__table) -> (schema, table) for Postgres"""
    schema, _, table = name.partition("__")
    return schema, table

PHASE_A_PLATFORM = {
    "public__gateway_requests", "public__control_audit_events", "auth__audit_log_entries",
    "public__workforce_events", "public__open_system_sync_events", "public__open_system_deployments",
    "cron__job_run_details", "net___http_response", "public__data_control_activity",
    "public__databricks_sync_audit", "public__e2e_autonomous_verification",
}
PHASE_A_OPS = {
    "public__open_tgate_audit_events", "public__open_tgate_workspace_audit_events",
    "public__open_tgate_worker_heartbeats", "auth__audit_log_entries",
    "public__databricks_sync_audit", "private__storage_usage_snapshots",
}
PHASES = {
    "A (logs & audit)": sorted(PHASE_A_PLATFORM | PHASE_A_OPS),
    "B (history content)": sorted((DELETE["open_platform"] - PHASE_A_PLATFORM)
                                  | (DELETE["open_operations"] - PHASE_A_OPS)),
    "C (auth sessions)": sorted(SESSION_CLEANUP),
}

PROJECT = {  # which mirror schema belongs to which Supabase project
    "open_platform": [t for t in rows if t.startswith("open_connect.open_platform.")],
    "open_operations": [t for t in rows if t.startswith("open_connect.open_operations.")],
}

def tables_in(proj, names):
    """names present in this project's mirror schema"""
    return [n for n in names if f"open_connect.{proj}.{n}" in PROJECT[proj]]

# Emit per (project, phase, table): each phase runs against that project's own database, and
# shared names (session tables, databricks_sync_audit, auth__audit_log_entries) must be
# emitted for BOTH projects — project_of() previously returned only the first match.
for proj in ("open_platform", "open_operations"):
    print(f"\n-- ############ PROJECT: {proj} — run these against this project's database ############")
    for phase, names in PHASES.items():
        present = tables_in(proj, names)
        if not present:
            continue
        print(f"-- ===== Phase {phase} =====")
        for name in present:
            s, t = to_pg(name)
            print(f"-- mirror rows: {rows.get(f'open_connect.{proj}.{name}', 0):,}")
            print(f"DELETE FROM {s}.{t};")
print("\n-- after each phase: VACUUM ANALYZE (psycopg2: conn.autocommit=True); re-run MEASURE_SQL; smoke-test apps")

## Phase 3 — Post-deletion

1. **Reclaim space:** `VACUUM ANALYZE` after each phase — via psycopg2 only with `conn.autocommit = True` (VACUUM cannot run inside a transaction); the Supabase SQL editor handles this itself. If the Supabase dashboard size stays high after vacuum, run `VACUUM FULL <table>` (locks the table — quiet window only) on the largest affected tables, or accept slower reclaim via autovacuum.
2. **Re-measure:** `MEASURE_SQL` again; compare against the free-tier limit from the dashboard.
3. **Smoke tests:** open-connect.site loads and authenticates; tgate worker heartbeat resumes and can still send/receive; scheduled jobs (`cron.job`) still fire.
4. **Close out:** update the memory file `/memories/projects/supabase-databricks-migration.md` with final sizes and deleted-row counts.

## Approval checklist (Gate 2 — reply in chat to authorize each step)

- [ ] Phase 0 results reviewed (live sizes + REVIEW decisions)
- [ ] Phase 1 fresh backup verified (row parity + private Docker Hub push)
- [ ] Phase A SQL approved
- [ ] Phase B SQL approved
- [ ] Phase C SQL approved (acknowledge: all users logged out)
- [ ] Phase D (if needed) reviewed per table

**Nothing above runs until Supabase credentials are provided and each phase is approved in chat.**